# 32a — Applications in Computer Vision

Corresponds to: Note 32 — Applications in Computer Vision

**Libraries introduced in this notebook:** No new libraries — PyTorch (`torch`, `torch.nn`) was introduced in Note 09.

**What this notebook demonstrates:**
1. IoU and mIoU computation — matching the Note 32 worked examples exactly
2. U-Net implementation in PyTorch — encoder, skip connections, decoder, output layer
3. Forward pass through U-Net on a random input — verifying output shape matches input spatial dimensions
4. Parameter count comparison — U-Net vs a plain encoder-decoder without skip connections

## Part 1 — IoU and mIoU

We compute IoU for the bounding box example and mIoU for the segmentation example from Note 32.

In [1]:
import torch

# ── IoU: bounding box example ─────────────────────────────────────────────────
pred  = (10, 12, 50, 48)   # x1, y1, x2, y2
gt    = (15, 10, 55, 45)

area_pred = (pred[2] - pred[0]) * (pred[3] - pred[1])
area_gt   = (gt[2]   - gt[0])   * (gt[3]   - gt[1])

x_left   = max(pred[0], gt[0])
x_right  = min(pred[2], gt[2])
y_top    = max(pred[1], gt[1])
y_bottom = min(pred[3], gt[3])

inter = max(0, x_right - x_left) * max(0, y_bottom - y_top)
union = area_pred + area_gt - inter
iou   = inter / union

print(f'Area pred:     {area_pred}')
print(f'Area gt:       {area_gt}')
print(f'Intersection:  {inter}')
print(f'Union:         {union}')
print(f'IoU:           {iou:.4f}')
print(f'TP at 0.5:     {iou >= 0.5}')

Area pred:     1440
Area gt:       1400
Intersection:  1155
Union:         1685
IoU:           0.6855
TP at 0.5:     True


Expected from Note 32: intersection = 1155, union = 1685, IoU ≈ 0.6860, TP = True.

In [2]:
# ── mIoU: segmentation example ────────────────────────────────────────────────
# 16 pixels, 2 classes
# true:      [0]*8 + [0]*2 + [1]*3 + [1]*3
# predicted: [0]*8 + [1]*2 + [0]*3 + [1]*3
y_true = torch.tensor([0]*8 + [0]*2 + [1]*3 + [1]*3)
y_pred = torch.tensor([0]*8 + [1]*2 + [0]*3 + [1]*3)

def class_iou(y_true, y_pred, c):
    tp = ((y_true == c) & (y_pred == c)).sum().item()
    fp = ((y_true != c) & (y_pred == c)).sum().item()
    fn = ((y_true == c) & (y_pred != c)).sum().item()
    return tp / (tp + fp + fn), tp, fp, fn

iou0, tp0, fp0, fn0 = class_iou(y_true, y_pred, 0)
iou1, tp1, fp1, fn1 = class_iou(y_true, y_pred, 1)
miou = (iou0 + iou1) / 2
pixel_acc = (y_true == y_pred).float().mean().item()

print(f'Class 0 — TP:{tp0} FP:{fp0} FN:{fn0}  IoU: {iou0:.4f}')
print(f'Class 1 — TP:{tp1} FP:{fp1} FN:{fn1}  IoU: {iou1:.4f}')
print(f'mIoU:          {miou:.4f}')
print(f'Pixel accuracy:{pixel_acc:.4f}')

Class 0 — TP:8 FP:3 FN:2  IoU: 0.6154
Class 1 — TP:3 FP:2 FN:3  IoU: 0.3750
mIoU:          0.4952
Pixel accuracy:0.6875


Expected from Note 32: IoU₀ ≈ 0.6154, IoU₁ = 0.3750, mIoU ≈ 0.4952, pixel accuracy = 0.6875.

## Part 2 — U-Net Implementation

We implement a compact U-Net in PyTorch with 4 encoder levels, a bottleneck, and 4 decoder levels with skip connections.

In [3]:
import torch.nn as nn

def conv_block(in_ch, out_ch):
    """Two 3×3 conv layers with ReLU — the basic U-Net building block."""
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True)
    )

class UNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2, base=64):
        super().__init__()
        # ── Encoder ───────────────────────────────────────────────────────────
        self.enc1 = conv_block(in_channels, base)
        self.enc2 = conv_block(base,     base*2)
        self.enc3 = conv_block(base*2,   base*4)
        self.enc4 = conv_block(base*4,   base*8)
        self.pool = nn.MaxPool2d(2)
        # ── Bottleneck ────────────────────────────────────────────────────────
        self.bottleneck = conv_block(base*8, base*16)
        # ── Decoder ───────────────────────────────────────────────────────────
        self.up4   = nn.ConvTranspose2d(base*16, base*8,  2, stride=2)
        self.dec4  = conv_block(base*16, base*8)   # *16 because of concat
        self.up3   = nn.ConvTranspose2d(base*8,  base*4,  2, stride=2)
        self.dec3  = conv_block(base*8,  base*4)
        self.up2   = nn.ConvTranspose2d(base*4,  base*2,  2, stride=2)
        self.dec2  = conv_block(base*4,  base*2)
        self.up1   = nn.ConvTranspose2d(base*2,  base,    2, stride=2)
        self.dec1  = conv_block(base*2,  base)
        # ── Output ────────────────────────────────────────────────────────────
        self.out   = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        # Bottleneck
        b  = self.bottleneck(self.pool(e4))
        # Decoder with skip connections
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

model = UNet(in_channels=1, num_classes=2, base=64)
total_params = sum(p.numel() for p in model.parameters())
print(f'U-Net total parameters: {total_params:,}')

U-Net total parameters: 31,030,658


## Part 3 — Forward Pass: Output Shape Verification

We pass a random $1 \times 1 \times 256 \times 256$ input through U-Net and verify the output has the same spatial dimensions.

In [4]:
torch.manual_seed(0)
x = torch.randn(1, 1, 256, 256)   # batch=1, channels=1, H=256, W=256

model.eval()
with torch.no_grad():
    out = model(x)

print(f'Input shape:  {tuple(x.shape)}')
print(f'Output shape: {tuple(out.shape)}')
print(f'Spatial dims preserved: {x.shape[2:] == out.shape[2:]}')

Input shape:  (1, 1, 256, 256)
Output shape: (1, 2, 256, 256)
Spatial dims preserved: True


The output should be `(1, 2, 256, 256)` — same spatial size as the input, with 2 class channels.

## Part 4 — Parameter Count: U-Net vs Plain Encoder-Decoder

We compare U-Net (with skip connections) against a plain encoder-decoder of the same depth and channel sizes but without skip connections. The skip connections add parameters only in the decoder conv blocks (wider input due to concatenation).

In [5]:
class PlainEncoderDecoder(nn.Module):
    """Same depth and channels as U-Net but no skip connections."""
    def __init__(self, in_channels=1, num_classes=2, base=64):
        super().__init__()
        self.enc1 = conv_block(in_channels, base)
        self.enc2 = conv_block(base,   base*2)
        self.enc3 = conv_block(base*2, base*4)
        self.enc4 = conv_block(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = conv_block(base*8, base*16)
        # Decoder: no concatenation — half the input channels vs U-Net
        self.up4  = nn.ConvTranspose2d(base*16, base*8, 2, stride=2)
        self.dec4 = conv_block(base*8,  base*8)
        self.up3  = nn.ConvTranspose2d(base*8,  base*4, 2, stride=2)
        self.dec3 = conv_block(base*4,  base*4)
        self.up2  = nn.ConvTranspose2d(base*4,  base*2, 2, stride=2)
        self.dec2 = conv_block(base*2,  base*2)
        self.up1  = nn.ConvTranspose2d(base*2,  base,   2, stride=2)
        self.dec1 = conv_block(base,    base)
        self.out  = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(self.up4(b))
        d3 = self.dec3(self.up3(d4))
        d2 = self.dec2(self.up2(d3))
        d1 = self.dec1(self.up1(d2))
        return self.out(d1)

plain = PlainEncoderDecoder(in_channels=1, num_classes=2, base=64)
plain_params = sum(p.numel() for p in plain.parameters())

print(f'U-Net parameters:              {total_params:,}')
print(f'Plain encoder-decoder params:  {plain_params:,}')
print(f'Extra params from skip conns:  {total_params - plain_params:,}')

U-Net parameters:              31,030,658
Plain encoder-decoder params:  27,897,218
Extra params from skip conns:  3,133,440


U-Net has more parameters in the decoder because skip connections double the input channels to each decoder conv block (concatenation). The extra parameters are the cost of the spatial detail recovery that skip connections provide.

## Summary

This notebook demonstrated:
1. IoU computation for bounding boxes and mIoU for pixel-level segmentation — matching Note 32 worked examples
2. U-Net implementation — encoder with max pooling, bottleneck, decoder with transposed convolutions and skip connections via concatenation
3. Forward pass verification — output spatial dimensions match input for a 256×256 input
4. Parameter count comparison — skip connections add parameters to decoder conv blocks by doubling their input channel count